# TreeAIBox Parameter Calibration

Run this notebook to analyze your point cloud and get recommended models and parameters.

**Instructions:**
1. **Cell 1** — Set your file path, sensor type, extent, and expected crown diameter
2. **Run All Cells** — The notebook will analyze your data and print recommended models + parameters
3. Copy the printed code snippets into your own workflow

In [ ]:
# ============================================================
# Cell 1: CONFIGURATION — edit these values
# ============================================================

POINT_CLOUD_PATH = r"C:\\Users\\mdshadman_amin\\Documents\\data\\opentopo_armycamp\\points.laz"

# Sensor: "ALS", "TLS", or "UAV"
SENSOR = "ALS"

# Extent: "landscape" (plot/forest with multiple trees) or "single_tree"
EXTENT = "landscape"

# Expected average crown diameter in meters (set None if unknown)
EXPECTED_CROWN_DIAMETER = 10.0

In [ ]:
# ============================================================
# Cell 2: ANALYZE POINT CLOUD
# ============================================================
import numpy as np
from scipy.spatial import cKDTree

# --- Load points ---
path = POINT_CLOUD_PATH.lower()
if path.endswith((".las", ".laz")):
    import laspy
    las = laspy.read(POINT_CLOUD_PATH)
    pts = np.column_stack([las.x, las.y, las.z])
elif path.endswith((".txt", ".csv")):
    import csv
    with open(POINT_CLOUD_PATH) as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = [[float(r[0]), float(r[1]), float(r[2])] for r in reader]
    pts = np.array(rows)
elif path.endswith((".ply",)):
    from plyfile import PlyData
    ply = PlyData.read(POINT_CLOUD_PATH)
    v = ply["vertex"]
    pts = np.column_stack([v["x"], v["y"], v["z"]])
else:
    raise ValueError(f"Unsupported file format: {POINT_CLOUD_PATH}")

# --- Basic stats ---
mn = pts.min(axis=0)
mx = pts.max(axis=0)
span = mx - mn
area = span[0] * span[1]

print(f"N points:        {len(pts):,}")
print(f"X range:         {mn[0]:.1f} → {mx[0]:.1f}  ({span[0]:.1f} m)")
print(f"Y range:         {mn[1]:.1f} → {mx[1]:.1f}  ({span[1]:.1f} m)")
print(f"Z range:         {mn[2]:.1f} → {mx[2]:.1f}  ({span[2]:.1f} m)")
print(f"Approx area:     {area:,.0f} m²  ({area / 10000:.2f} ha)")

# --- Point density & spacing ---
density = len(pts) / area
tree = cKDTree(pts[:, :2])
dd, _ = tree.query(pts[:, :2], k=2)
avg_spacing = np.median(dd[:, 1])

print(f"Density:         ~{density:.0f} pts/m²")
print(f"Median spacing:  {avg_spacing * 100:.1f} cm")

# --- Store for later cells ---
stats = {
    "n_points": len(pts),
    "span": span,
    "area": area,
    "z_range": span[2],
    "density": density,
    "spacing": avg_spacing,
}

In [ ]:
# ============================================================
# Cell 3: MODEL SELECTION
# ============================================================

# --- Filtering model ---
filter_models = {
    "ALS": {
        # Pick resolution based on density
        "high":  "treefiltering_als_esegformer3D_128_15cm(GPU3GB)",   # >10 pts/m²
        "med":   "treefiltering_als_esegformer3D_128_50cm(GPU3GB)",   # 2-10 pts/m²
        "low":   "treefiltering_als_esegformer3D_128_80cm(GPU3GB)",   # <2 pts/m²
    },
    "TLS": "treefiltering_tls_esegformer3D_128_8cm(GPU3GB)",
    "UAV": "treefiltering_uav_esegformer3D_128_12cm(GPU3GB)",
}

if SENSOR == "ALS":
    if density > 10:
        filter_model = filter_models["ALS"]["high"]
    elif density > 2:
        filter_model = filter_models["ALS"]["med"]
    else:
        filter_model = filter_models["ALS"]["low"]
else:
    filter_model = filter_models[SENSOR]

# --- Urban filtering (ALS only, optional) ---
urban_filter_model = "urbanfiltering_als_esegformer3D_112_30cm(GPU3GB)" if SENSOR == "ALS" else None

# --- Tree location model ---
treeloc_models = {
    "ALS": "treeisonet_als_reclamation_treeloc_esegformer3D_128_10cm(GPU4GB)",
    "TLS": "treeisonet_tls_boreal_treeloc_esegformer3D_128_10cm(GPU3GB)",
    "UAV": "treeisonet_uav_mixedwood_treeloc_esegformer3D_128_10cm(GPU3GB)",
}
treeloc_model = treeloc_models[SENSOR]

# --- Stem classification model (TLS/UAV only) ---
stemcls_models = {
    "TLS": {
        "high": "treeisonet_tls_boreal_stemcls_esegformer3D_128_4cm(GPU3GB)",   # >100 pts/m²
        "low":  "treeisonet_tls_boreal_stemcls_esegformer3D_128_10cm(GPU3GB)",  # ≤100 pts/m²
    },
    "UAV": "treeisonet_uav_mixedwood_stemcls_esegformer3D_128_8cm(GPU3GB)",
}
if SENSOR == "TLS":
    stemcls_model = stemcls_models["TLS"]["high"] if density > 100 else stemcls_models["TLS"]["low"]
elif SENSOR == "UAV":
    stemcls_model = stemcls_models["UAV"]
else:
    stemcls_model = None  # ALS has no stem classification

# --- Tree offset model ---
treeoff_models = {
    "ALS": "treeisonet_als_reclamation_treeoff_esegformer3D_128_10cm(GPU4GB)",
}
treeoff_model = treeoff_models.get(SENSOR, None)

# --- Crown offset model (TLS/UAV) ---
crownoff_models = {
    "TLS": "treeisonet_tls_boreal_crownoff_esegformer3D_128_15cm(GPU4GB)",
    "UAV": "treeisonet_uav_mixedwood_crownoff_esegformer3D_128_15cm(GPU4GB)",
}
crownoff_model = crownoff_models.get(SENSOR, None)

# --- Wood classification models (TLS only) ---
if SENSOR == "TLS":
    if density > 500:
        woodcls_stem_model = "woodcls_stem_tls_esegformer3D_128_4cm(GPU3GB)"
        woodcls_branch_model = "woodcls_branch_tls_esegformer3D_128_2.5cm(GPU3GB)"
    else:
        woodcls_stem_model = "woodcls_stem_tls_esegformer3D_128_10cm(GPU3GB)"
        woodcls_branch_model = "woodcls_branch_tls_segformer3D_112_4cm(GPU6GB)"
else:
    woodcls_stem_model = None
    woodcls_branch_model = None

# --- Print selected models ---
print("=" * 60)
print(f"RECOMMENDED MODELS  (sensor={SENSOR}, extent={EXTENT})")
print(f"  density={density:.0f} pts/m², spacing={avg_spacing*100:.1f} cm")
print("=" * 60)
print(f"  Filter:          {filter_model}")
if urban_filter_model:
    print(f"  Urban filter:    {urban_filter_model}")
print(f"  Tree location:   {treeloc_model}")
if stemcls_model:
    print(f"  Stem cls:        {stemcls_model}")
if treeoff_model:
    print(f"  Tree offset:     {treeoff_model}")
if crownoff_model:
    print(f"  Crown offset:    {crownoff_model}")
if woodcls_stem_model:
    print(f"  Wood cls (stem): {woodcls_stem_model}")
    print(f"  Wood cls (branch): {woodcls_branch_model}")

In [ ]:
# ============================================================
# Cell 4: PARAMETER CALIBRATION
# ============================================================

# Crown diameter: measured or estimated
cd = EXPECTED_CROWN_DIAMETER if EXPECTED_CROWN_DIAMETER else span[0] * 0.15  # rough fallback

# --- tree_filtering parameters ---
if_bottom_only = True if EXTENT == "landscape" else False
# bottom_only=True uses 2D sliding window (faster for landscapes)
# bottom_only=False uses 3D sliding window (better for individual trees)

# --- tree_location parameters ---
if_stem = SENSOR in ("TLS", "UAV")  # stem detection only for close-range sensors
# cutoff_thresh: confidence threshold for tree detection (lower = more detections)
if SENSOR == "ALS":
    cutoff_thresh = 0.8   # ALS: higher threshold (less noise, clearer canopy peaks)
elif SENSOR == "UAV":
    cutoff_thresh = 1.0   # UAV: moderate
else:
    cutoff_thresh = 1.0   # TLS: moderate

# custom_resolution: override model's native voxel size (None = use model default)
# Only set if your data density differs significantly from training data
custom_resolution = None

# --- post_peak_extraction parameters ---
# K: number of neighbors for local peak search
K = max(3, min(10, int(cd / avg_spacing / 5)))  # scale with crown/spacing ratio
K = min(K, 10)

# max_gap: max distance (m) to merge nearby peak candidates
max_gap = cd * 0.03  # ~3% of crown diameter

# min_rad: minimum crown radius for a valid tree
min_rad = cd * 0.1   # 10% of crown diameter

# nms_thresh: non-maximum suppression IoU threshold
nms_thresh = 0.3     # 0.3 works well for most cases; increase for very dense canopies

# --- stem_clustering parameters ---
stem_min_res = max(0.03, avg_spacing * 3)       # ~3x point spacing
stem_max_isolated = max(0.2, avg_spacing * 10)   # isolated point threshold

# --- crown_clustering parameters ---
crown_min_res = max(0.10, avg_spacing * 5)       # coarser for crown segmentation
crown_K = 5                                       # neighbors for graph cut
crown_reg_strength = 1.0                          # regularization (higher = smoother clusters)
crown_max_isolated = max(0.2, cd * 0.03)          # isolated point threshold

# --- Print all parameters ---
print("=" * 60)
print("RECOMMENDED PARAMETERS")
print("=" * 60)

print(f"\n── tree_filtering ──")
print(f"  model:          {filter_model}")
print(f"  if_bottom_only: {if_bottom_only}")
print(f"  use_cuda:       True  (set False if no GPU)")

print(f"\n── tree_location ──")
print(f"  model:          {treeloc_model}")
print(f"  if_stem:        {if_stem}")
print(f"  cutoff_thresh:  {cutoff_thresh}")
print(f"  custom_res:     {custom_resolution}")

print(f"\n── post_peak_extraction ──")
print(f"  K:              {K}")
print(f"  max_gap:        {max_gap:.2f} m")
print(f"  min_rad:        {min_rad:.2f} m")
print(f"  nms_thresh:     {nms_thresh}")

if stemcls_model:
    print(f"\n── stem_clustering ──")
    print(f"  min_res:              {stem_min_res:.3f} m")
    print(f"  max_isolated_distance: {stem_max_isolated:.3f} m")

if crownoff_model or treeoff_model:
    print(f"\n── crown_clustering ──")
    print(f"  min_res:              {crown_min_res:.3f} m")
    print(f"  K:                    {crown_K}")
    print(f"  reg_strength:         {crown_reg_strength}")
    print(f"  max_isolated_distance: {crown_max_isolated:.3f} m")